In [9]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
import joblib

RANDOM_STATE = 42

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


In [10]:


# 🔧 ADJUST THESE PATHS
LSMS_PATH = "/Users/jjburrell/Downloads/send/lsms_panel_allvars_2010_2024_MASTER.csv"   # or .csv
EE_PATH   = "/Users/jjburrell/Downloads/send/EE_harvest_ml.csv"        # you uploaded this here

print("🔹 Loading LSMS from:", LSMS_PATH)
lsms_df = pd.read_csv(LSMS_PATH)
print("LSMS shape:", lsms_df.shape)
print("LSMS columns:", lsms_df.columns.tolist())
display(lsms_df.head())

print("\n🔹 Loading EE (harvest features) from:", EE_PATH)
ee_df = pd.read_csv(EE_PATH)
print("EE shape:", ee_df.shape)
print("EE columns:", ee_df.columns.tolist())
display(ee_df.head())


🔹 Loading LSMS from: /Users/jjburrell/Downloads/send/lsms_panel_allvars_2010_2024_MASTER.csv
LSMS shape: (1350540, 18)
LSMS columns: ['ea_id_obs', 'ea_id_merge', 'lat_modified', 'lon_modified', 'year', 'month', 'ym', 'precip', 'tmax', 'tmin', 'soil_moist', 'NDVI', 'EVI', 'GCVI', 'GDD', 'KDD', 'source_file', 'date']


/var/folders/kh/hkkfd68s6mb8b11z27mn1b4h0000gn/T/ipykernel_60456/886788105.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  lsms_df = pd.read_csv(LSMS_PATH)


,ea_id_obs,ea_id_merge,lat_modified,lon_modified,year,month,ym,precip,tmax,tmin,soil_moist,NDVI,EVI,GCVI,GDD,KDD,source_file,date
0,1000001.0,10101088800910,14.258308,37.809654,2010,1,2010-01,0.000000,32.919459,16.665438,0.096080,0.154746,0.100348,0.760022,458.565904,0.000000,lsms_panel_allvars_2010.csv,2010-01-01
1,1000001.0,10101088800910,14.258308,37.809654,2010,2,2010-02,0.000000,34.638923,18.247248,0.095678,0.183497,0.107605,1.179610,460.406400,0.000000,lsms_panel_allvars_2010.csv,2010-02-01
2,1000001.0,10101088800910,14.258308,37.809654,2010,3,2010-03,7.860581,35.808196,20.337499,0.100096,0.150189,0.095727,0.921962,560.258272,5.886407,lsms_panel_allvars_2010.csv,2010-03-01
3,1000001.0,10101088800910,14.258308,37.809654,2010,4,2010-04,21.636782,37.869912,23.465852,0.096824,NaN,NaN,NaN,620.036469,24.269983,lsms_panel_allvars_2010.csv,2010-04-01
4,1000001.0,10101088800910,14.258308,37.809654,2010,5,2010-05,28.362386,35.904275,22.227520,0.126311,NaN,NaN,NaN,591.042810,4.607831,lsms_panel_allvars_2010.csv,2010-05-01



🔹 Loading EE (harvest features) from: /Users/jjburrell/Downloads/send/EE_harvest_ml.csv
EE shape: (6187, 115)
EE columns: ['country', 'wave', 'season', 'ea_id_obs', 'ea_id_merge', 'lat_modified', 'lon_modified', 'EVI_0', 'EVI_1', 'EVI_10', 'EVI_11', 'EVI_2', 'EVI_3', 'EVI_4', 'EVI_5', 'EVI_6', 'EVI_7', 'EVI_8', 'EVI_9', 'GCVI_0', 'GCVI_1', 'GCVI_10', 'GCVI_11', 'GCVI_2', 'GCVI_3', 'GCVI_4', 'GCVI_5', 'GCVI_6', 'GCVI_7', 'GCVI_8', 'GCVI_9', 'GDD_0', 'GDD_1', 'GDD_10', 'GDD_11', 'GDD_2', 'GDD_3', 'GDD_4', 'GDD_5', 'GDD_6', 'GDD_7', 'GDD_8', 'GDD_9', 'KDD_0', 'KDD_1', 'KDD_10', 'KDD_11', 'KDD_2', 'KDD_3', 'KDD_4', 'KDD_5', 'KDD_6', 'KDD_7', 'KDD_8', 'KDD_9', 'NDVI_0', 'NDVI_1', 'NDVI_10', 'NDVI_11', 'NDVI_2', 'NDVI_3', 'NDVI_4', 'NDVI_5', 'NDVI_6', 'NDVI_7', 'NDVI_8', 'NDVI_9', 'precip_0', 'precip_1', 'precip_10', 'precip_11', 'precip_2', 'precip_3', 'precip_4', 'precip_5', 'precip_6', 'precip_7', 'precip_8', 'precip_9', 'soil_moist_0', 'soil_moist_1', 'soil_moist_10', 'soil_moist_11', '

,country,wave,season,ea_id_obs,ea_id_merge,lat_modified,lon_modified,EVI_0,EVI_1,EVI_10,...,tmin_10,tmin_11,tmin_2,tmin_3,tmin_4,tmin_5,tmin_6,tmin_7,tmin_8,tmin_9
0,Ethiopia,1.0,1.0,1000002,010101088801601,14.353816,37.890876,0.089968,0.094474,NaN,...,21.529931,18.626803,15.601708,16.983214,17.626778,17.751088,18.513651,19.375479,20.144021,22.150149
1,Ethiopia,1.0,1.0,1000004,010102088801403,14.288590,38.210252,0.146575,0.219179,NaN,...,12.445024,12.334106,14.770980,15.751398,15.951006,16.626983,18.247125,18.088582,15.755054,13.680830
2,Ethiopia,1.0,1.0,1000005,010103010100106,14.109761,38.473835,0.179362,0.260539,NaN,...,12.855313,12.313958,13.367965,14.076063,15.024848,15.060475,16.045982,17.447881,17.068529,14.872832
3,Ethiopia,1.0,1.0,1000007,010103088801804,13.844084,38.480325,0.187695,0.321498,0.142435,...,12.932498,12.497424,15.429626,16.186867,16.223335,16.562163,18.381762,17.399455,15.388825,12.731499
4,Ethiopia,1.0,1.0,1000010,010105088800204,14.086276,37.983187,0.178514,0.206576,NaN,...,13.992304,14.170070,16.032736,16.730414,17.000939,17.722803,19.720051,19.712134,17.436311,15.474077


In [11]:
# Keep only rows with valid IDs and NDVI
lsms_required = ["ea_id_obs", "ea_id_merge", "lat_modified", "lon_modified", "NDVI"]
lsms_df = lsms_df.dropna(subset=lsms_required)
print("LSMS after dropping NA in ID/NDVI:", lsms_df.shape)

# Aggregate NDVI per location (ea_id + lat/lon)
group_keys = ["ea_id_obs", "ea_id_merge", "lat_modified", "lon_modified"]
agg_lsms = (
    lsms_df
    .groupby(group_keys, dropna=False)
    .agg(target_ndvi=("NDVI", "mean"))
    .reset_index()
)

print("Aggregated LSMS NDVI shape:", agg_lsms.shape)
display(agg_lsms.head())


LSMS after dropping NA in ID/NDVI: (969740, 18)
Aggregated LSMS NDVI shape: (6609, 5)


,ea_id_obs,ea_id_merge,lat_modified,lon_modified,target_ndvi
0,1000001.0,10101088800910,14.258308,37.809654,0.313387
1,1000002.0,10101088801601,14.353816,37.890876,0.264088
2,1000003.0,10102088801010,14.375913,38.154888,0.347796
3,1000004.0,10102088801403,14.288590,38.210252,0.303872
4,1000005.0,10103010100106,14.109761,38.473835,0.337030


In [12]:
merge_keys = ["ea_id_obs", "ea_id_merge", "lat_modified", "lon_modified"]
print("Merging LSMS NDVI with EE features on:", merge_keys)

df = pd.merge(
    agg_lsms,
    ee_df,
    on=merge_keys,
    how="inner"
)

df = df.dropna()
print("✅ Final merged dataset shape:", df.shape)
display(df.head())

# Save merged dataset for reproducibility
df.to_csv("merged_lsms_ee_ndvi.csv", index=False)
print("Saved → merged_lsms_ee_ndvi.csv")


Merging LSMS NDVI with EE features on: ['ea_id_obs', 'ea_id_merge', 'lat_modified', 'lon_modified']
✅ Final merged dataset shape: (563, 116)


,ea_id_obs,ea_id_merge,lat_modified,lon_modified,target_ndvi,country,wave,season,EVI_0,EVI_1,...,tmin_10,tmin_11,tmin_2,tmin_3,tmin_4,tmin_5,tmin_6,tmin_7,tmin_8,tmin_9
0,3000825.0,844,16.118035,-3.656784,0.179856,Mali,1.0,1.0,0.125335,0.131666,...,16.343764,16.874052,25.110637,27.158679,25.963412,28.378014,29.131180,27.670994,23.831759,19.247175
1,3000825.0,844,16.118035,-3.656784,0.179856,Mali,2.0,1.0,0.126953,0.148004,...,19.440345,14.090363,25.534730,27.016195,26.389776,28.029975,29.061032,29.542180,25.167768,23.255727
2,3000826.0,845,16.172841,-3.598720,0.175492,Mali,1.0,1.0,0.125055,0.135686,...,15.767732,16.240219,24.151237,26.924221,25.937266,28.294980,28.784761,26.993635,22.834260,18.315866
3,3000826.0,845,16.172841,-3.598720,0.175492,Mali,2.0,1.0,0.121614,0.128600,...,19.029980,13.661495,24.863116,26.916688,26.414397,28.009858,28.855156,28.969309,24.425845,22.728051
4,3000827.0,846,16.147991,-3.609649,0.220370,Mali,1.0,1.0,0.122781,0.122741,...,16.894044,15.878503,27.178861,26.006254,28.401137,29.152990,27.617696,23.836384,19.262485,16.370759


Saved → merged_lsms_ee_ndvi.csv


In [13]:
TARGET_COL = "target_ndvi"

# ID/meta columns to drop from features
id_cols = [
    "country", "wave", "season",
    "ea_id_obs", "ea_id_merge",
    "lat_modified", "lon_modified"
]

id_cols = [c for c in id_cols if c in df.columns]

drop_cols = id_cols + [TARGET_COL]
feature_cols = [c for c in df.columns if c not in drop_cols]

print("Target:", TARGET_COL)
print("Num features:", len(feature_cols))
print("Example feature cols:", feature_cols[:15])

X_full = df[feature_cols].reset_index(drop=True)
y_full = df[TARGET_COL].reset_index(drop=True)

# Grouping for spatial CV: ea_id_merge
groups_full = df["ea_id_merge"].reset_index(drop=True)

# Wave-based train/test split from EE
if "wave" not in df.columns:
    raise ValueError("Column 'wave' not found in merged df; required for train/test split.")

max_wave = int(df["wave"].max())
train_mask = df["wave"] < max_wave
test_mask  = df["wave"] == max_wave

X_train = X_full[train_mask].reset_index(drop=True)
y_train = y_full[train_mask].reset_index(drop=True)
groups_train = groups_full[train_mask].reset_index(drop=True)

X_test  = X_full[test_mask].reset_index(drop=True)
y_test  = y_full[test_mask].reset_index(drop=True)
meta_test = df[test_mask].reset_index(drop=True)

print(f"Train waves: {sorted(df.loc[train_mask, 'wave'].unique())}")
print(f"Test wave: {max_wave}")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Train countries:", df.loc[train_mask, "country"].unique())
print("Test countries:", df.loc[test_mask, "country"].unique())


Target: target_ndvi
Num features: 108
Example feature cols: ['EVI_0', 'EVI_1', 'EVI_10', 'EVI_11', 'EVI_2', 'EVI_3', 'EVI_4', 'EVI_5', 'EVI_6', 'EVI_7', 'EVI_8', 'EVI_9', 'GCVI_0', 'GCVI_1', 'GCVI_10']


/var/folders/kh/hkkfd68s6mb8b11z27mn1b4h0000gn/T/ipykernel_60456/3864684163.py:33: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train = X_full[train_mask].reset_index(drop=True)


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).